# create example files

## imports

In [ ]:
# if the package is not installed in your python environment, run this to execute the notebook directly from inside the GitHub repository
%cd -q ..

In [ ]:
from itertools import product

import numpy as np
import asdf
import pandas as pd
import sympy

import weldx

from weldx import Q_
from weldx import LocalCoordinateSystem as LCS
from weldx import WXRotation
from weldx import utility as ut
from weldx import geometry as geo
from weldx import measurement as msm
from weldx import transformations as tf
from weldx.core import MathematicalExpression

In [ ]:
import itertools


def dict_product(dicts):
    """
    >>> list(dict_product(dict(number=[1,2], character='ab')))
    [{'character': 'a', 'number': 1},
     {'character': 'a', 'number': 2},
     {'character': 'b', 'number': 1},
     {'character': 'b', 'number': 2}]
    """
    return (dict(zip(dicts, x)) for x in itertools.product(*dicts.values()))

## Timestamps

In [ ]:
timestamp_list = [
    pd.Timestamp("2020-01-01 12:00:00"),
    pd.Timestamp("2020-02-01 12:00:00"),
]

## grooves

In [ ]:
# generate test grooves
from weldx.welding.groove.iso_9692_1 import _create_test_grooves

groove_dict = _create_test_grooves()
for k in ["dv_groove2", "dv_groove3", "du_groove2", "du_groove3", "du_groove4"]:
    groove_dict.pop(k, None)
groove_list = list(groove_dict.values())
groove_list = [groove_list[0], groove_list[1], groove_list[2], groove_list[3]]

## seam length

In [ ]:
seam_dict = {"100mm": Q_(100, "mm"), "300mm": Q_(300, "mm")}
seam_list = seam_dict.values()

## processes

In [ ]:
from weldx.core import TimeSeries as TS
from weldx.welding.processes import GmawProcess

# Note: For some reasons, using integers in Q_ fails upon ASDF reading !
params_spray = dict(
    wire_feedrate=Q_(10.0, "m/min"),
    voltage=TS(data=Q_([40.0, 20.0], "V"), time=Q_([0.0, 10.0], "s")),
    impedance=Q_(10.0, "percent"),
    characteristic=Q_(5, "V/A"),
)
process_spray = GmawProcess(
    "spray", "CLOOS", "Quinto", params_spray, tag="CLOOS/spray_arc"
)

params_pulse = dict(
    wire_feedrate=Q_(10.0, "m/min"),
    pulse_voltage=Q_(40.0, "V"),
    pulse_duration=Q_(5.0, "ms"),
    pulse_frequency=Q_(100.0, "Hz"),
    base_current=Q_(60.0, "A"),
)
process_pulse = GmawProcess(
    "pulse",
    "CLOOS",
    "Quinto",
    params_pulse,
    tag="CLOOS/pulse",
    meta={"modulation": "UI"},
)

process_dict = {"spray": process_spray, "pulse": process_pulse}

process_list = process_dict.values()

## shielding gasses

In [ ]:
from weldx.asdf.tags.weldx.aws.process.gas_component import GasComponent
from weldx.asdf.tags.weldx.aws.process.shielding_gas_for_procedure import (
    ShieldingGasForProcedure,
)
from weldx.asdf.tags.weldx.aws.process.shielding_gas_type import ShieldingGasType

flows = [Q_(10, "l/min"), Q_(20, "l/min")]
argons = [Q_(82, "percent"), Q_(100, "percent")]


def make_gas(flow, ar):
    co2 = Q_(100, "percent") - ar
    gas_comp = [
        GasComponent("argon", ar),
        GasComponent("carbon dioxide", co2),
    ]
    gas_type = ShieldingGasType(gas_component=gas_comp, common_name="SG")
    return ShieldingGasForProcedure(
        use_torch_shielding_gas=True,
        torch_shielding_gas=gas_type,
        torch_shielding_gas_flowrate=flow,
    )


gas_list = [make_gas(*l) for l in product(*[flows, argons])]

## measurements

In [ ]:
I_freqs = [Q_(0.0, "1/s"), Q_(10.0, "1/s")]
U_freqs = [Q_(0.0, "1/s"), Q_(10.0, "1/s")]

I_amps = [Q_(100, "A")]
U_amps = [Q_(5, "V")]

I_biases = [Q_(300, "A")]
U_biases = [Q_(20, "V")]

msm_params = list(product(*[I_freqs, U_freqs, I_amps, U_amps, I_biases, U_biases]))

## generate tree

In [ ]:
from asdf.tags.core import Software


def build_msms_data(seam_length, I_freq, U_freq, I_amp, U_amp, I_bias, U_bias):
    time_end = str(seam_length / Q_(10, "mm/s"))
    time = pd.timedelta_range(start="0s", end=time_end, freq="5ms")

    # current data
    I_ts = ut.sine(f=I_freq, amp=I_amp, bias=I_bias)
    I = I_ts.interp_time(time)
    I["time"] = I["time"]

    current_data = msm.Data(name="Welding current", data=I)

    # voltage data
    U_ts = ut.sine(f=U_freq, amp=U_amp, bias=U_bias, phase=Q_(0.1, "rad"))
    U = U_ts.interp_time(time)
    U["time"] = U["time"]

    voltage_data = msm.Data(name="Welding voltage", data=U)

    return current_data, voltage_data

def build_msm(current_data, voltage_data):
    HKS_sensor = msm.GenericEquipment(name="HKS P1000-S3")
    BH_ELM = msm.GenericEquipment(name="Beckhoff ELM3002-0000")
    twincat_scope = Software(name="Beckhoff TwinCAT ScopeView", version="3.4.3143")

    src_current = msm.Source(
        name="Current Sensor",
        output_signal=msm.Signal(signal_type="analog", unit="V", data=None),
        error=msm.Error(Q_(0.1, "percent")),
    )

    HKS_sensor.sources = []
    HKS_sensor.sources.append(src_current)

    from weldx.core import MathematicalExpression

    [a, x, b] = sympy.symbols("a x b")
    current_AD_func = MathematicalExpression(a * x + b)
    current_AD_func.set_parameter("a", Q_(32768.0 / 10.0, "1/V"))
    current_AD_func.set_parameter("b", Q_(0.0, ""))

    current_AD_transform = msm.DataTransformation(
        name="AD conversion current measurement",
        input_signal=src_current.output_signal,
        output_signal=msm.Signal("digital", "", data=None),
        error=msm.Error(Q_(0.01, "percent")),
        func=current_AD_func,
    )

    BH_ELM.data_transformations = []
    BH_ELM.data_transformations.append(current_AD_transform)

    # define current output calibration expression and transformation
    current_calib_func = MathematicalExpression(a * x + b)
    current_calib_func.set_parameter("a", Q_(1000.0 / 32768.0, "A"))
    current_calib_func.set_parameter("b", Q_(0.0, "A"))

    current_calib_transform = msm.DataTransformation(
        name="Calibration current measurement",
        input_signal=current_AD_transform.output_signal,
        output_signal=msm.Signal("digital", "A", data=current_data),
        error=msm.Error(0.0),
        func=current_calib_func,
        meta=twincat_scope,
    )

    welding_current_chain = msm.MeasurementChain(
        name="welding current measurement chain",
        data_source=src_current,
        data_processors=[current_AD_transform, current_calib_transform],
    )

    welding_current = msm.Measurement(
        name="welding current measurement",
        data=[current_data],
        measurement_chain=welding_current_chain,
    )

    # In[10]:

    src_voltage = msm.Source(
        name="Voltage Sensor",
        output_signal=msm.Signal("analog", "V", data=None),
        error=msm.Error(Q_(0.1, "percent")),
    )

    HKS_sensor.sources.append(src_voltage)

    # define AD conversion expression and transformation step
    [a, x, b] = sympy.symbols("a x b")
    voltage_ad_func = MathematicalExpression(a * x + b)
    voltage_ad_func.set_parameter("a", Q_(32768.0 / 10.0, "1/V"))
    voltage_ad_func.set_parameter("b", Q_(0.0, ""))

    voltage_AD_transform = msm.DataTransformation(
        name="AD conversion voltage measurement",
        input_signal=src_voltage.output_signal,
        output_signal=msm.Signal("digital", "", data=None),
        error=msm.Error(Q_(0.01, "percent")),
        func=voltage_ad_func,
    )

    HKS_sensor.data_transformations.append(voltage_AD_transform)

    # define voltage output calibration expression and transformation
    voltage_calib_func = MathematicalExpression(a * x + b)
    voltage_calib_func.set_parameter("a", Q_(100.0 / 32768.0, "V"))
    voltage_calib_func.set_parameter("b", Q_(0.0, "V"))

    voltage_calib_transform = msm.DataTransformation(
        name="Calibration voltage measurement",
        input_signal=voltage_AD_transform.output_signal,
        output_signal=msm.Signal("digital", "V", data=voltage_data),
        error=msm.Error(0.0),
        func=voltage_calib_func,
        meta=twincat_scope,
    )

    welding_voltage_chain = msm.MeasurementChain(
        name="welding voltage measurement chain",
        data_source=src_voltage,
        data_processors=[voltage_AD_transform, voltage_calib_transform],
    )

    welding_voltage = msm.Measurement(
        name="welding voltage measurement",
        data=[voltage_data],
        measurement_chain=welding_voltage_chain,
    )

    equipment = [HKS_sensor, BH_ELM]
    measurements = [welding_current, welding_voltage]
    welding_current = (current_calib_transform.output_signal,)
    welding_voltage = (voltage_calib_transform.output_signal,)

    return equipment, measurements, welding_current, welding_voltage

In [ ]:
def build_csm(seam_length):
    # create a linear trace segment a the complete weld seam trace
    trace_segment = geo.LinearHorizontalTraceSegment(seam_length)
    trace = geo.Trace(trace_segment)

    # crete a new coordinate system manager with default base coordinate system
    csm = weldx.transformations.CoordinateSystemManager("base")

    # add the workpiece coordinate system
    csm.add_cs(
        coordinate_system_name="workpiece",
        reference_system_name="base",
        lcs=trace.coordinate_system,
    )

    tcp_start_point = Q_([5.0, 0.0, 2.0], "mm")
    tcp_end_point = Q_([-5.0, 0.0, 2.0], "mm") + np.append(
        seam_length, Q_([0, 0], "mm")
    )

    v_weld = Q_(10, "mm/s")
    s_weld = (tcp_end_point - tcp_start_point)[0]  # length of the weld
    t_weld = s_weld / v_weld

    t_start = pd.Timedelta("0s")
    t_end = pd.Timedelta(str(t_weld.to_base_units()))

    rot = WXRotation.from_euler(seq="x", angles=180, degrees=True)

    coords = [tcp_start_point.magnitude, tcp_end_point.magnitude]

    tcp_wire = LCS(coordinates=coords, orientation=rot, time=[t_start, t_end])

    # add the workpiece coordinate system
    csm.add_cs(
        coordinate_system_name="tcp_wire",
        reference_system_name="workpiece",
        lcs=tcp_wire,
    )

    tcp_contact = LCS(coordinates=[0, 0, -10])

    # add the workpiece coordinate system
    csm.add_cs(
        coordinate_system_name="tcp_contact",
        reference_system_name="tcp_wire",
        lcs=tcp_contact,
    )

    return csm

In [ ]:
def build_tree(reference_timestamp, groove_shape, seam_length, welding_process, shielding_gas, msm_param):
    groove_shape = groove_shape[0]
    
    csm = build_csm(seam_length)
    
    current_data, voltage_data = build_msms_data(seam_length, *msm_param)
    equipment, measurements, welding_current, welding_voltage = build_msm(current_data, voltage_data)

    tree = dict(
        reference_timestamp=reference_timestamp,
        geometry=dict(groove_shape=groove_shape, seam_length=seam_length),
        process=dict(welding_process=welding_process, shielding_gas=shielding_gas),
        equipment=equipment,
        measurements=measurements,
        welding_current=welding_current,
        welding_voltage=welding_voltage,
        coordinate_systems=csm,
    )

    return tree

In [ ]:
i = 0
for elements in product(*[timestamp_list, groove_list, seam_list, process_list, gas_list, msm_params]):
    i+=1
    filename = f"KISA/data/asdf_ref_{i:05d}.asdf"
    
    tree = build_tree(*elements)
    
    with asdf.AsdfFile(tree) as ff:
        ff.write_to(filename, all_array_compression="zlib")
        
print(i)